# Fund Liquidity Stress and LMT Workflow

This notebook assesses the selected fund's ability to meet redemption pressure through three analytical layers:

**1. Fund and Portfolio Setup**  
Profile inputs, portfolio holdings, liquidity classification by settlement speed, and estimated days to liquidate.

**2. Point-in-Time RMP Stress**  
Deterministic redemption shocks (typically 5%, 15%, 25%) from the Risk Management Policy. Tests whether the fund can meet a single, defined redemption shock against available liquid assets.

**3. Dynamic Redemption Path**  
A separate 12-month simulation based on investor-type behaviour assumptions from calibration and computed investor weights. Used to illustrate how liquidity management tools (gates, swing pricing, suspension) respond to evolving redemption pressure, deferred redemptions, backlog accumulation, and conditional contagion feedback.

**Note:** The dynamic path is independent from the RMP scenario percentages. It represents a forward-looking simulation layer, not a direct application of RMP shocks.

References for the dynamic redemption modeling can be found [here](../../docs/notebooks_notes/liquidity_references.md).

# Setup

In [ ]:
# connect to SQL database
from src.data.database   import get_engine
ENGINE  = get_engine()

# fund to be analysed
FUND = "UCITS_Balanced"

# In this repo all computation refer to the same date
from src.config import VALUATION_DATE
VALUATION_DATE

In [ ]:
# Query a risk-ready DataFrame with all columns needed for
# risk computtion including liquidty risk
from src.data.enrichment import get_risk_ready_df
risk_df = get_risk_ready_df(ENGINE, FUND, VALUATION_DATE)
nav = risk_df['market_value_eur'].sum()

## 1. Fund Liquidity Terms and Policy Inputs

The fund profile and liquidity monitoring inputs are queried from the electronic fund records.

> Note: `Pct ADV` defines the assumed daily trading capacity. The stress window defines the liquidity monitoring horizon.


In [ ]:
# Display fund overview combined with liquidity monitoring parameters
import src.ui.liquidity_calibration_display as lcd
lcd.display_fund_liquidity_overview(
    fund_id=FUND,
    engine=ENGINE,
    export_id="01",
)

## 2. Portfolio Liquidity Profile

This section estimates how quickly the selected fund’s holdings could be converted into cash under the project’s liquidity classification. This creates the asset-side liquidity view used later in the stress test.


> Note: The stress window is shown as a monitoring parameter in Section 1. The current liquidity profile uses %ADV (= participation rate) to estimate time to liquidate each position and does not apply the stress window as a calculation constraint.Days-to-liquidate estimates position-level liquidation timelines. Liquidity buckets classify assets by expected settlement speed.

In [ ]:
# Liquidity profile — plot only
from src.ui.liquidity_plot import plot_liquidity_profile
from src.risk.risk_utils import compute_liquidity_profile
from src.data.reference_data import load_liquidity_calibration_inputs

# Load pct_adv from policy configuration
calib_for_adv = load_liquidity_calibration_inputs(FUND)
liquidity_pct_adv = calib_for_adv['stress_assumptions']['pct_adv']

liq = compute_liquidity_profile(risk_df, pct_adv=liquidity_pct_adv)
bucket_full = liq['bucket_full']
risk_df_liq = liq['risk_df_liq']

x = plot_liquidity_profile(
    bucket_full, 
    FUND, 
    metric='pct_nav_abs', 
    valuation_date=VALUATION_DATE, 
    export_id="02",
    )

## 3. Investor Base and Redemption Scenario Inputs
Liability-side inputs are loaded from the investor registry, to compute concentration. The RMP redemption scenarios define the point-in-time redemption shocks applied to the valuation-date NAV before LMT mechanics are considered.


In [ ]:
# Load investor registers and calibration inputs
from src.data.reference_data import load_investor_and_calibration_data

# Load calibration data for UCITS_Balanced
data = load_investor_and_calibration_data(FUND)
investor_inputs = data['investor_inputs']
calibration_inputs = data['calibration_inputs']
calibration_config = data['calibration_config']

# Display top 3 investors
lcd.display_top_investors(
    investor_inputs, 
    FUND, 
    VALUATION_DATE, 
    top_n=3, 
    export_id="03",
    )

In [ ]:
lcd.display_investor_base(
    investor_inputs,
    fund_id=FUND,
    valuation_date=VALUATION_DATE,
    export_id="04",
    folder_suffix='_liquidity',
)

## 4. Point-in-Time Redemption Stress

The point-in-time redemption exercise applies a selected redemption scenario to the fund’s valuation-date portfolio. It compares the redemption amount with the liquid assets available under the project liquidity buckets. 

In [ ]:
# Redemption stress — use existing function
import src.ui.print_html_utils as phtml
from src.data.reference_data import get_stress_testing_params
from src.pipeline.liquidity_policy import load_redemption_scenarios_from_rmp

# Load redemption scenarios from RMP
REDEMPTION_SCENARIOS = load_redemption_scenarios_from_rmp(FUND)

# Get stress testing params
calibration_inputs = data['calibration_inputs']
params = get_stress_testing_params(FUND, calibration_inputs, redemption_scenario="Large")
notice_period_days = params['notice_days']

# Display using existing function
phtml.display_redemption_stress(
    FUND,
    notice_period_days,
    REDEMPTION_SCENARIOS,
    nav,
    risk_df_liq,
    valuation_date=VALUATION_DATE,
    export_id="05",
)


## 5. Dynamic Redemption Path — Before LMT Effects

This section simulates the redemption path over 12 months without liquidity management tools applied. It shows how redemption pressure evolves month-by-month when no gates, swing pricing, or suspension mechanics are active.

The purpose is to establish a baseline: raw redemption demand, resulting fund-level outflows, and NAV evolution in an unmanaged scenario. This provides the counterfactual against which LMT effects can be measured.

The dynamic path uses the same monthly redemption schedule as the point-in-time stress scenarios but projects it forward across future dealing dates, allowing for backlog and contagion effects to accumulate.

In [ ]:
# Display LMT parameters from calibration
lcd.display_lmt_parameters(calibration_inputs, FUND, export_id="06")

In [ ]:
# LMT Trigger Analysis — Before and After LMT Effects
from src.pipeline.lmt_trigger_analysis import (
    build_lmt_analysis_inputs,
)
from src.computation.liquidity import lmt_trigger_analysis
from IPython.display import HTML

# Build LMT inputs from calibration
calibration_config = data['calibration_config']
lmt_inputs = build_lmt_analysis_inputs(
    FUND,
    calibration_inputs,
    calibration_config,
)

lmt_config = lmt_inputs['lmt_config']
liquid_pct = lmt_config['liquid_pct']
schedule = lmt_inputs['schedule']


**Investor Redemption Model**

Investor weights are computed from the investor registry using NAV percentages and mapped to the liquidity calibration categories. Redemption rates are sourced from the liquidity calibration inputs.

Weights by investor category:
- Registry investor types (Pension Plan, Platform, Insurance, Retail, Family Office) are mapped to liquidity model categories (Institutional, Retail)
- Weights are the sum of NAV percentages for all investors mapped to each category
- These computed weights drive the monthly redemption schedule

This path is separate from the risk management policy redemption scenarios, which define point-in-time policy stress shocks.


In [ ]:
investors_enriched = lmt_inputs['lmt_params']['investors_enriched']

lcd.display_investor_redemption_inputs(
    investors_enriched,
    fund_id=FUND,
    valuation_date=VALUATION_DATE,
    export_id="07",  
)


<small>
> Note: The redemption rates on RMP scenarios and the investor-type calibration assumptions serve different purposes. RMP scenarios define policy-level point-in-time redemption shocks. Calibration assumptions define investor-type redemption behaviour used in the dynamic path simulation, and uses average of redemption rates weighted by investor type.


In [ ]:
# Scenario 1: Dynamic path WITHOUT LMT effects (all None to disable)
result_before_lmt = lmt_trigger_analysis(
    nav=nav,
    liquid_pct=liquid_pct,
    gate_threshold=None,
    swing_threshold=None,
    redemption_schedule=schedule,
    consecutive_gate_for_suspension=None,
    contagion_multiplier=1.0,
    apply_contagion=False,
)


In [ ]:
# After-LMT combined view (compact)
lcd.plot_lmt_analysis(
    result_before_lmt, 
    FUND, 
    VALUATION_DATE,
    scenario_label="No LMT",
    show_inset=True,
    export_id="08",
)

## 6. Dynamic Redemption Path After LMTs

The after-LMT run applies the fund’s LMT settings to the calibrated redemption path. Results show the amount paid, deferred and carried forward as backlog, including any post-gate contagion generated by the model.


In [ ]:
# Scenario 2: Dynamic path WITH LMT effects (policy thresholds)
result_after_lmt = lmt_trigger_analysis(
    nav=nav,
    liquid_pct=liquid_pct,
    gate_threshold=lmt_config['gate_threshold'],
    swing_threshold=lmt_config['swing_threshold'],
    redemption_schedule=schedule,
    consecutive_gate_for_suspension=lmt_config['consec_gate'],
    contagion_multiplier=lmt_config['contagion'],
    apply_contagion=True,
)

In [ ]:
# After-LMT combined view (compact)
lcd.plot_lmt_analysis(
    result_after_lmt, 
    FUND, 
    VALUATION_DATE,
    scenario_label="After LMT",
    show_inset=True,
    export_id="09",
)


)
